# 01 Get the data
Downloads the raw data **once** and saves it in `data/raw/` with the fixed date `DOWNLOAD_DATE` (set in `helpers.py`) in the file name. If a file already exists it is not downloaded again, so every later notebook works on the same frozen data. It also writes two small tables that are kept in the repository: `download_log.csv` (SHA-256 of every raw file) and `index_manifest.csv` (one row per index).

* **ParlGov** (elections and cabinets, development version of 4 Nov 2023): https://parlgov.org/data-info/ . 
* **National stock indices**: daily closes from Yahoo Finance via `yfinance`, from 1 Jan 1995 up to the day BEFORE `DOWNLOAD_DATE` (the end date is excluded, so the file never contains a partly completed trading day). 

In [1]:
import sys
from pathlib import Path

roots = [Path.cwd(), Path("/files"), Path.home()]
code_dir = next(p for r in roots for p in [r, *r.glob("**/elections_project/code")] if (p / "helpers.py").exists())
sys.path.insert(0, str(code_dir))
print("Working in", code_dir)

import hashlib
from datetime import date

import pandas as pd

from helpers import DOWNLOAD_DATE, PARLGOV_TABLES, PARLGOV_URL, RAW, TICKERS, raw_file

RAW.mkdir(parents=True, exist_ok=True)
print("Frozen download date:", DOWNLOAD_DATE, "| today:", date.today().isoformat())

Working in /files/mywork/elections_project/code
Frozen download date: 2026-11-19 | today: 2026-11-19


## ParlGov: elections and cabinets

In [2]:
for table in PARLGOV_TABLES:
    path = RAW / f"parlgov_{table}_{DOWNLOAD_DATE}.csv"
    if path.exists():
        print(f"{path.name}: already downloaded, not downloaded again")
        continue
    df = pd.read_csv(PARLGOV_URL + table + ".csv")
    df.to_csv(path, index=False)
    print(f"{table}: {len(df):,} rows, {df.shape[1]} columns -> {path.name}")

view_election: 8,997 rows, 16 columns -> parlgov_view_election_2026-11-19.csv
view_cabinet: 12,640 rows, 19 columns -> parlgov_view_cabinet_2026-11-19.csv


## Stock indices

In [3]:
path = RAW / f"index_prices_{DOWNLOAD_DATE}.csv"
if path.exists():
    print(f"{path.name}: already downloaded, not downloaded again")
else:
    import yfinance as yf
    prices = yf.download(list(TICKERS.values()), start="1995-01-01", end=DOWNLOAD_DATE,
                         auto_adjust=True, progress=False)["Close"]
    prices.index.name = "date"
    prices.to_csv(path)
    print(f"{prices.shape[1]} indices, {len(prices):,} days -> {path.name}")

23 indices, 8,312 days -> index_prices_2026-11-19.csv


## Download log and index manifest
The SHA-256 identifies the exact file that was analysed. A later download from Yahoo can differ (Yahoo may correct past values), and this log is how that is detected.

In [4]:
import platform

try:
    import yfinance as yf
    yf_version = yf.__version__
except ImportError:
    yf_version = "not installed in this run"

files = [raw_file("parlgov_" + t) for t in PARLGOV_TABLES] + [raw_file("index_prices")]
log = pd.DataFrame({
    "file": [f.name for f in files],
    "download_date": DOWNLOAD_DATE,
    "bytes": [f.stat().st_size for f in files],
    "rows": [len(pd.read_csv(f)) for f in files],
    "sha256": [hashlib.sha256(f.read_bytes()).hexdigest() for f in files],
    "yfinance": yf_version,
    "pandas": pd.__version__,
    "python": platform.python_version(),
})
log.to_csv(RAW.parent / "download_log.csv", index=False)
log[["file", "rows", "sha256"]]

,file,rows,sha256
0,parlgov_view_election_2026-11-19.csv,8997,d129b05b3f69671bdcb6beba4312aa5c4bc00a34e71439...
1,parlgov_view_cabinet_2026-11-19.csv,12640,eb2dda61669a86fa8737ce70eb2b61713303c74d47ae65...
2,index_prices_2026-11-19.csv,8312,c53b2d934c5158c884975ddd3ab052e3d86b0612b7c495...


In [5]:
# Return type of the series, as far as it is documented in the research notes for this project; "unverified" otherwise.
RETURN_TYPE = {
    "^OMXTGI": "gross (total return)", "^OMXVGI": "gross (total return)", "^NZ50": "gross (total return)",
    "TA35.TA": "gross (labelled GTR by TASE)", "^OMX": "price", "^OMXH25": "price", "PSI20.LS": "price",
}
prices = pd.read_csv(raw_file("index_prices"), index_col="date", parse_dates=True)
manifest = pd.DataFrame([{
    "country": c, "ticker": t,
    "first_date": prices[t].first_valid_index().date(),
    "last_date": prices[t].last_valid_index().date(),
    "n_days": int(prices[t].notna().sum()),
    "return_type": RETURN_TYPE.get(t, "unverified"),
} for c, t in TICKERS.items()]).sort_values("first_date")
manifest.to_csv(RAW.parent / "index_manifest.csv", index=False)
manifest

,country,ticker,first_date,last_date,n_days,return_type
11,ISR,TA35.TA,1995-01-02,2026-11-18,6302,gross (labelled GTR by TASE)
1,AUT,^ATX,1995-01-02,2026-11-18,7927,unverified
14,NLD,^AEX,1995-01-02,2026-11-18,8138,unverified
5,DEU,^GDAXI,1995-01-02,2026-11-18,8082,unverified
6,ESP,^IBEX,1995-01-02,2026-11-18,8084,unverified
8,GBR,^FTSE,1995-01-03,2026-11-18,8054,unverified
7,FRA,^FCHI,1995-01-03,2026-11-18,8121,unverified
0,AUS,^AXJO,1995-01-03,2026-11-18,8060,unverified
3,CAN,^GSPTSE,1995-01-03,2026-11-18,8012,unverified
2,BEL,^BFX,1995-01-03,2026-11-18,8122,unverified
